In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy tqdm

!pip install torch==2.3.1 bitsandbytes==0.43.1 --quiet
!pip install transformers==4.44.2 peft==0.11.1 accelerate==0.34.2 --quiet

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets scikit-learn pandas numpy tqdm matplotlib cvss



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 36.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.9 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 80.9 MB/s eta 0:00:00:00:0100:01
ERROR: pip's de

In [2]:
# 1. Uninstall possibly incompatible preinstalled packages
!pip uninstall -y torch torchvision torchaudio transformers accelerate datasets

# 2. Install compatible PyTorch + torchvision + transformers for Kaggle GPU (CUDA 11.8)
# Use PyTorch built wheels for CUDA 11.8
!pip install --index-url https://download.pytorch.org/whl/cu118 \
    torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0+cu118

# 3. Install compatible HF stack
!pip install transformers==4.42.4 datasets==2.20.0 accelerate==0.33.0 scikit-learn==1.5.2 sentencepiece shap tqdm


Found existing installation: torch 2.6.0+cu118
Uninstalling torch-2.6.0+cu118:
  Successfully uninstalled torch-2.6.0+cu118
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: accelerate 0.34.2
Uninstalling accelerate-0.34.2:
  Successfully uninstalled accelerate-0.34.2
Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 398.4 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 91.7 MB/s eta 0:0

In [3]:
import torch, torchvision, transformers
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    !nvidia-smi


Torch: 2.1.0+cu118
Torchvision: 0.16.0+cu118
Transformers: 4.42.4
CUDA available: True
Wed Nov 12 07:45:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |          

In [4]:
import pandas as pd
import re, html
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer


# Load dataset

DATA_PATH = "/kaggle/input/csv-ds/cve.csv"   
df = pd.read_csv(DATA_PATH)

print(" Loaded data shape:", df.shape)
print("Columns:", df.columns.tolist())

# Ensure required columns exist
if "summary" not in df.columns or "cvss" not in df.columns:
    raise ValueError("Expected columns: ['summary', 'cvss', 'pub_date']")


# Clean data

def score_to_severity(score):
    if score == 0.0:
        return "None"
    elif score <= 3.9:
        return "Low"
    elif score <= 6.9:
        return "Medium"
    elif score <= 8.9:
        return "High"
    else:
        return "Critical"

df["severity"] = df["cvss"].apply(score_to_severity)
df = df.dropna(subset=["summary", "cvss"]).drop_duplicates(subset=["summary"])

# Encode severity labels
le = LabelEncoder()
df["severity_label"] = le.fit_transform(df["severity"])


# Safe sort (no errors param)

if "pub_date" in df.columns:
    try:
        df["pub_date"] = pd.to_datetime(df["pub_date"], errors="coerce")
        df = df.sort_values("pub_date").reset_index(drop=True)
    except Exception as e:
        print(" Could not sort by pub_date:", e)
        df = df.reset_index(drop=True)
else:
    print(" 'pub_date' column not found, skipping time-based sort.")
    df = df.reset_index(drop=True)


# Train / Validation / Test Split

n = len(df)
train_end = int(n * 0.8)
val_end = int(n * 0.9)
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


# Tokenization setup

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text.strip())
    return text

def preprocess_function(batch):
    texts = [clean_text(x) for x in batch["summary"]]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=MAX_LEN)

def safe_map(dataset):
    remove_cols = [c for c in dataset.column_names if c in ["summary", "pub_date"]]
    return dataset.map(preprocess_function, batched=True, remove_columns=remove_cols)


# Convert pandas → Hugging Face datasets

hf_train = Dataset.from_pandas(train_df)
hf_val = Dataset.from_pandas(val_df)
hf_test = Dataset.from_pandas(test_df)

# Apply preprocessing safely
hf_train = safe_map(hf_train)
hf_val = safe_map(hf_val)
hf_test = safe_map(hf_test)


# Add numeric + label columns

for ds_name, ds, df_part in zip(["train", "val", "test"], [hf_train, hf_val, hf_test], [train_df, val_df, test_df]):
    new_cols = {
        "base_score": df_part["cvss"].astype(float).tolist(),
        "severity_label": df_part["severity_label"].astype(int).tolist()
    }
    for col in ["base_score", "severity_label"]:
        if col in ds.column_names:
            ds = ds.remove_columns([col])
    ds = ds.add_column("base_score", new_cols["base_score"])
    ds = ds.add_column("severity_label", new_cols["severity_label"])

    if ds_name == "train":
        hf_train = ds
    elif ds_name == "val":
        hf_val = ds
    else:
        hf_test = ds


# Create classification and regression variants

hf_train_cls = hf_train.rename_column("severity_label", "labels")
hf_val_cls = hf_val.rename_column("severity_label", "labels")
hf_test_cls = hf_test.rename_column("severity_label", "labels")

hf_train_reg = hf_train.rename_column("base_score", "labels")
hf_val_reg = hf_val.rename_column("base_score", "labels")
hf_test_reg = hf_test.rename_column("base_score", "labels")


# Set torch format

hf_train_cls.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
hf_val_cls.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
hf_train_reg.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
hf_val_reg.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(" Tokenization complete! Ready for training.")


 Loaded data shape: (89660, 13)
Columns: ['Unnamed: 0', 'mod_date', 'pub_date', 'cvss', 'cwe_code', 'cwe_name', 'summary', 'access_authentication', 'access_complexity', 'access_vector', 'impact_availability', 'impact_confidentiality', 'impact_integrity']
Train: 69945 | Val: 8743 | Test: 8744


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/69945 [00:00<?, ? examples/s]

Map:   0%|          | 0/8743 [00:00<?, ? examples/s]

Map:   0%|          | 0/8744 [00:00<?, ? examples/s]

 Tokenization complete! Ready for training.


In [6]:

#  CVSS Severity Classification Model Training (Transformers v5+ Compatible)


# Imports
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import os
import torch


# Config

MODEL_NAME = "bert-base-uncased"
OUTPUT_DIR = "/kaggle/working"   # or "./" if local
EPOCHS = 3
BATCH_SIZE = 8
LR = 2e-5


# Model Setup

num_labels = len(le.classes_)  # 'le' from LabelEncoder
cls_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)


# Metrics

def compute_metrics_cls(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "f1": f1}


# Training Arguments (v5 compatible)

training_args_cls = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "cls_results"),
    eval_strategy="epoch",               #  changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none",                   # disable W&B, etc.
    remove_unused_columns=False,
    logging_dir=os.path.join(OUTPUT_DIR, "logs"),
    logging_strategy="steps",
    logging_steps=100,
)


# Trainer

trainer_cls = Trainer(
    model=cls_model,
    args=training_args_cls,
    train_dataset=hf_train_cls,   #  classification dataset
    eval_dataset=hf_val_cls,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics_cls,
)


# Train Model

print("==  TRAINING CLASSIFICATION MODEL ==")
trainer_cls.train()


# Save Model & Tokenizer

save_path = os.path.join(OUTPUT_DIR, "cvss_classification_model")
trainer_cls.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f" Model saved to: {save_path}")


# Evaluate Model

eval_results = trainer_cls.evaluate()
print("\n==  Validation Results ==")
for k, v in eval_results.items():
    print(f"{k}: {v:.4f}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


==  TRAINING CLASSIFICATION MODEL ==


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.569800,0.648869,0.739449,0.732319
2,0.458800,0.677940,0.737276,0.738958
3,0.415800,0.719949,0.744481,0.745607


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


 Model saved to: /kaggle/working/cvss_classification_model


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '



==  Validation Results ==
eval_loss: 0.7199
eval_accuracy: 0.7445
eval_f1: 0.7456
eval_runtime: 95.1680
eval_samples_per_second: 91.8690
eval_steps_per_second: 5.7480
epoch: 3.0000


In [7]:

# CVSS Base Score Regression Model Training (Transformers v5+ Compatible)


from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import os
import torch


# Config

MODEL_NAME = "bert-base-uncased"
OUTPUT_DIR = "/kaggle/working"   # or "./" if local
EPOCHS = 3
BATCH_SIZE = 8
LR = 2e-5


# Model Setup

reg_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
reg_model.config.problem_type = "regression"


# Metrics

def compute_metrics_reg(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = preds.squeeze()
    preds = np.clip(preds, 0.0, 10.0)  # CVSS scores are between 0–10
    mae = mean_absolute_error(labels, preds)
    rmse = mean_squared_error(labels, preds, squared=False)
    return {"mae": mae, "rmse": rmse}


# Training Arguments (v5+ compatible)

training_args_reg = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "reg_results"),
    eval_strategy="epoch",           #  Transformers v5 syntax
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="mae",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
    logging_dir=os.path.join(OUTPUT_DIR, "logs_reg"),
    logging_strategy="steps",
    logging_steps=100,
)


# Trainer Setup

trainer_reg = Trainer(
    model=reg_model,
    args=training_args_reg,
    train_dataset=hf_train_reg,    #  regression dataset (from your preprocessing)
    eval_dataset=hf_val_reg,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics_reg,
)


# Train Regression Model

print("\n==  TRAINING REGRESSION MODEL ==")
trainer_reg.train()


# Save Regression Model & Tokenizer

save_path_reg = os.path.join(OUTPUT_DIR, "cvss_regression_model")
trainer_reg.save_model(save_path_reg)
tokenizer.save_pretrained(save_path_reg)
print(f" Regression model saved to: {save_path_reg}")


# Evaluate Regression Model

eval_results_reg = trainer_reg.evaluate()
print("\n==  Regression Model Evaluation Results ==")
for k, v in eval_results_reg.items():
    print(f"{k}: {v:.4f}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



==  TRAINING REGRESSION MODEL ==


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Mae,Rmse
1,1.258000,1.740110,0.960495,1.319118
2,1.031200,1.444001,0.827055,1.201583
3,0.807000,1.549751,0.842556,1.244824


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

 Regression model saved to: /kaggle/working/cvss_regression_model


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '



==  Regression Model Evaluation Results ==
eval_loss: 1.7401
eval_mae: 0.9605
eval_rmse: 1.3191
eval_runtime: 95.5850
eval_samples_per_second: 91.4680
eval_steps_per_second: 5.7230
epoch: 3.0000


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
